# 03 · Feature engineering from data already in hand

Follow-up to `02_spatial_holdout.ipynb`. Same base table, splits, models and metrics
(now in `holdout.py`); this notebook adds feature groups one at a time and keeps what
helps.

**Groups tested**

1. **DCAD building systems.** Area-weighted shares of apartment buildings with central
   heat, no heating system (heaters or none), central AC, window or wall AC, wood frame,
   pier-and-beam foundation, frame exterior; years since remodel; functional and external
   depreciation. Dallas's life hazards include lack of heat and lack of AC.
2. **Spatial context (label-free).** Multitenant properties and units within 0.5 km and
   1.5 km, distance to downtown.
3. **Owner track record.** The bottom-10 rate of the owner's other *inspected* properties,
   grouped two ways: DCAD owner name, and owner mailing address (single-asset LLCs under
   one sponsor usually share a mailing address). Plus sponsor portfolio size.
4. **Nearby outcomes.** Bottom-10 rate of the 10 nearest inspected properties and their
   distance.

Groups 3 and 4 use labels, so they're built inside each split from the training rows
only. Under the Chelsea premise that's information the city would have: inspections in
the target area reveal owners and neighborhoods. Training rows get 5-fold out-of-fold
owner rates (leave-one-out leaks a row's own label to tree models), and a training row
never counts itself among its neighbors.

## Findings

Run 2026-10-01. Outputs are cleared before commit; the numbers below are from that run.

Ensemble PR-AUC for `bottom10`. Test prevalence is 0.163 on the Chelsea mirror, 0.197 on the
random split and 0.196 on grouped CV.

| Configuration | Chelsea mirror | Random 70/30 | Grouped CV by tract |
|---|---|---|---|
| 02 baseline | 0.270 | 0.398 | 0.312 |
| + DCAD systems | 0.265 | 0.385 | 0.323 |
| + spatial context | 0.281 | 0.401 | 0.311 |
| + owner record | 0.285 | 0.441 | 0.331 |
| + nearby outcomes | 0.294 | 0.404 | 0.327 |
| all four | 0.310 | 0.436 | 0.328 |
| all but DCAD (carried forward) | 0.308 | 0.438 | 0.328 |

- **Owner track record** helps on all three splits. Its gain is largest on the random split
  (0.40 → 0.44), where more of an owner's other properties are in training. On the Chelsea
  mirror only 77 test properties share an owner name with the training area, but 39% of
  those 77 are bottom-10, against 15% of the rest. The mailing-address sponsor key reaches
  more properties than the owner name does.
- **Nearby outcomes** (bottom-10 rate of the 10 nearest training properties) help the two
  spatial splits. It is the top feature by permutation importance.
- **DCAD building systems** do not help, so they are dropped from later notebooks. These are
  heating and AC type, frame, foundation, remodel year, and functional and external
  depreciation. Most apartment buildings in DCAD have central heat and air (87–89% by
  area), which leaves little variation.
- **Spatial context** (multitenant density, distance to downtown) adds about a point on the
  mirror and nothing elsewhere.
- **First-inspection target:** the owner record also helps here (grouped CV 0.133 →
  0.163). The mirror's test area holds about 100 positives, so differences of a point or
  two there are noise.
- **Leak check:** with shuffled labels the label-based features score at prevalence (0.18–0.20
  against 0.19–0.20). The fold construction does not leak.
- The DCAD commercial improvement table's value columns are entirely null in the platform
  silver, so no value features come from it. Notebook 04 takes values from the account
  table instead.

In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

import features as f
import holdout as h

warnings.filterwarnings("ignore", message="Skipping features without any observed values")
pd.set_option("display.float_format", "{:.3f}".format)

df = h.load_base()
DCAD = f.dcad_detail(df)
CONTEXT = f.spatial_context(df)
df = pd.concat([df, DCAD, CONTEXT, f.owner_keys(df)[["log_sponsor_portfolio"]]], axis=1)
print("rows:", len(df))
print("DCAD detail coverage:", f"{DCAD.share_central_heat.notna().mean():.0%}")
print(DCAD.mean().round(3).to_string())

## Ablation

In [ ]:
B = h.BASELINE_FEATURES
DCAD_COLS, CONTEXT_COLS = list(DCAD.columns), list(CONTEXT.columns)
CONFIGS = {
    "02 baseline": (B, None),
    "+ DCAD systems": (B + DCAD_COLS, None),
    "+ spatial context": (B + CONTEXT_COLS, None),
    "+ owner record": (B + ["log_sponsor_portfolio"], f.owner_track_record),
    "+ nearby outcomes": (B, f.nearby_outcomes),
    "all four": (
        B + DCAD_COLS + CONTEXT_COLS + ["log_sponsor_portfolio"],
        f.combine(f.owner_track_record, f.nearby_outcomes),
    ),
    "all but DCAD": (
        B + CONTEXT_COLS + ["log_sponsor_portfolio"],
        f.combine(f.owner_track_record, f.nearby_outcomes),
    ),
}
results = pd.concat(
    [
        h.run(df, t, feats, fold_features=ff, label=name)
        for t in h.TARGETS
        for name, (feats, ff) in CONFIGS.items()
    ],
    ignore_index=True,
)

In [ ]:
for t in h.TARGETS:
    r = results[results.target == t]
    print(f"\n=== {t}: PR-AUC (test prevalence: "
          + ", ".join(f"{s} {r[r.split == s].prevalence_test.iloc[0]:.3f}" for s in h.SPLITS) + ")")
    print(h.compare(r).round(3).to_string())

In [ ]:
for t in h.TARGETS:
    print(f"\n=== {t}: share of bottom-10 found in the top 20%")
    print(h.compare(results[results.target == t], "found@20%").round(3).to_string())

## Leak check for the label-based features

In [ ]:
# With shuffled labels the owner and neighbour features carry no real signal, so PR-AUC
# should fall to prevalence. If it doesn't, the fold features are leaking.
rng = np.random.default_rng(h.SEED)
shuffled = df.assign(bottom10=rng.permutation(df.bottom10.to_numpy()))
chk = h.run(
    shuffled, "bottom10", B + ["log_sponsor_portfolio"],
    fold_features=f.combine(f.owner_track_record, f.nearby_outcomes),
    which={"Chelsea mirror", "Random 70/30"}, label="shuffled",
)
print(chk[["split", "ranker", "prevalence_test", "PR-AUC"]].to_string(index=False))

## What carries the combined model (Chelsea mirror, `bottom10`)

In [ ]:
FEATS = B + DCAD_COLS + CONTEXT_COLS + ["log_sponsor_portfolio"]
FOLD = f.combine(f.owner_track_record, f.nearby_outcomes)
est, X_te, y_te = h.fit_split(df, "bottom10", FEATS, "Chelsea mirror", "Random forest", fold_features=FOLD)
r = permutation_importance(est, X_te, y_te, scoring="average_precision", n_repeats=20,
                           random_state=h.SEED, n_jobs=h.N_JOBS)
imp = pd.Series(r.importances_mean, index=X_te.columns).sort_values(ascending=False)
print(imp.head(15).round(4).to_string())